# 9b · Grounding — **full-article judge** (twin of nb9)

Same question as nb9 — *is every asserted n-ary node and relation actually supported by the source
article?* — with the evidence plumbing removed. nb9 hands the judge a per-assertion **evidence bundle**
it assembles itself (quote → sentence expansion → n-ary pooling → adjacent-sentence coverage repair).
This notebook hands it **the whole article, once per call**, and asks it to find the support itself.

**Why.** On art1 nb9 ships 128,854 chars of evidence bundles for a 4,745-char article — the same 32
sentences re-sent with every assertion that touches them — for ~47k input tokens across 8 calls. The
full article plus the bare assertions is ~12.6k tokens. Cheaper, and it retires three knobs
(`MAX_SENTENCE_CHARS`, `MAX_CONTEXT_GAP`, n-ary evidence pooling) whose combined effect was to widen
what the judge reads by a median 2.3× — so nb9's number is really *"supported somewhere in an expanded
window"*, which is not the claim its framing makes.

**What this measures instead.** `grounded` here means **supported by the article**, full stop. That is
the more standard and more defensible claim — and it will read *higher* than nb9's, because assertions
whose support sits a sentence away from the cited quote now pass. That is a definitional change, not a
looser judge: do not report the two rates as a before/after improvement.

**What is kept, and what is new**
- **Kept:** the deterministic citation check (no API calls). It no longer gates the judge — a quote the
  extractor invented can't hide, but it also can't suppress an assertion the article does support. It
  is reported alongside every verdict.
- **Kept:** n-ary sibling context on the assertion line. That disambiguates what an edge *means*
  (`--[hasIncrease]--> 62%` is the Paris-region sub-figure of the France hub), which is a separate
  question from what the judge is allowed to read.
- **New:** the judge must return the **verbatim sentence** it grounded each assertion on. That sentence
  is then located in the article deterministically (a fabricated quote is caught), and compared with
  what the extractor cited — giving **citation precision**: *did the extractor quote the sentence that
  actually supports its own assertion?* nb9 could not ask this, because it chose the evidence itself.

Two numbers come out of one pass: **grounding rate** (is the graph faithful to the article) and
**citation precision** (is its provenance honest).

In [14]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [ ]:
# ---- WHICH MODEL'S RESULTS ---------------------------------------------------------------
# Graphs are read from and judgments written to results/<RESULTS_MODEL>/. Choose any subfolder
# that exists there: "gemini-2.5-flash", "qwen3-14b", "qwen3-32b", "llama33-70b", ...
RESULTS_MODEL = "gemini-2.5-flash"

# Which model acts as the JUDGE. One judge for every model's graphs is what makes the arms
# comparable — but it must not be the model that WROTE the graphs being judged, or the grounding
# rate is a self-report. Gemini's own graphs therefore want an open-weight judge (needs a vLLM
# server + tunnel; see cluster/README.md), and the open-weight arms want Gemini.
# Section 8 measures whether the judge you pick is actually any good before you believe its number.
BACKEND = "qwen3-32b-think"

# ---- WHICH ARTICLES -----------------------------------------------------------------------
# This list defines the run. Section 6 grounds every id in it (WORKERS at a time), section 6's
# summary pools over it, and section 7 diffs all of them against nb9. Sections 1-5 are the
# step-by-step walkthrough of ONE article — the first in the list — so put whichever you want to
# inspect at the front. Keep this list the same as nb9's for the comparison to be like-for-like.
#   None            -> every article under results/<RESULTS_MODEL>/ that has GRAPH_NAME
#   ["art5", ...]   -> exactly these, in this order (ids without the graph are reported and skipped)
ARTICLES = None
# ARTICLES = ["art5", "art1", "art12"]
# ------------------------------------------------------------------------------------------

# --- Config -------------------------------------------------------------------------------
from pathlib import Path

GEMINI_MODEL = "gemini-2.5-flash"          # the grounding judge

GRAPH_NAME   = "nary_graph_dedup_main_val_r1_extended_consolidated.json"   # input graph (per article, e2e/)
                                                           # ^ the tag of the run being judged (see results/RUNS.md)
OUT_NAME     = "nary_graph_grounded_fulltext.json"          # written next to it
TRACK_DIR    = "e2e"                                           # results/<id>/<TRACK_DIR>/

# Judge knobs. The article is re-sent with every batch (~1.2k tokens for a news story), so batching
# costs almost nothing on the input side — it exists to keep the OUTPUT list short, which is where a
# long run of verdicts degrades or hits MAX_TOKENS. 60 assertions ≈ 2k output tokens.
JUDGE_BATCH   = 60         # 20 for a THINKING judge: it reasons about every assertion in the
                           # batch before emitting any of them, so a long batch degrades
JUDGE_RETRIES = 3          # attempts per batch before giving up (transient API/JSON errors)
JUDGE_RETRY_BACKOFF = 5    # seconds between retries; linear backoff = BACKOFF * attempt

# Batch knob. Articles are grounded independently of one another, so the batch cell at the end can
# run several at once: WORKERS is how many ARTICLES are in flight (each is its own sequence of judge
# calls, so the API sees up to WORKERS concurrent requests). 1 = the old sequential loop, which is
# what you want while debugging. Gemini paid tier: 4-8 is comfortable; free tier (10 RPM): 1-2.
WORKERS = 4

# Citation-check knobs. These NO LONGER decide what the judge sees — they only score the extractor's
# quotes against the article, and locate the judge's returned sentence.
FUZZY_THRESHOLD    = 0.85  # difflib ratio; >= this against the best window -> "partial", else "missing"
MAX_SPAN_CHARS     = 400   # elided quote whose fragments span more than this = not one quote -> missing
MAX_SENTENCE_CHARS = 600   # cap when widening a located quote to its sentence (reporting only)

RESULTS_DIR = ROOT / "results" / RESULTS_MODEL   # results/<model>/<id>/<track>/
ARTICLE_DIR = ROOT / "dataset" / "articles"
GLOBAL_DIR  = RESULTS_DIR / "_global"            # run-level outputs (summary, judge sample, usage)

# --- resolve ARTICLES -> the concrete ids this run touches ---------------------------------
import re as _re
def _natkey(s):                                  # so art10 sorts after art9, not after art1
    return [int(t) if t.isdigit() else t for t in _re.split(r"(\d+)", str(s))]

def resolve_articles(ids=None):
    """-> the article ids that actually have the input graph. None = discover every one that does."""
    has_graph = lambda a: (RESULTS_DIR / a / TRACK_DIR / GRAPH_NAME).exists()
    if ids is None:
        return sorted((p.name for p in RESULTS_DIR.iterdir() if p.is_dir() and has_graph(p.name)),
                      key=_natkey)
    missing = [a for a in ids if not has_graph(a)]
    if missing:
        print(f"!! skipped — no {TRACK_DIR}/{GRAPH_NAME}: {missing}")
    return [a for a in ids if has_graph(a)]

ARTICLE_IDS = resolve_articles(ARTICLES)
assert ARTICLE_IDS, f"no article under {RESULTS_DIR} has {TRACK_DIR}/{GRAPH_NAME}"
ARTICLE_ID = ARTICLE_IDS[0]                      # the one sections 1-5 walk through

print(f"{len(ARTICLE_IDS)} article(s): {ARTICLE_IDS}")
print(f"walkthrough: {ARTICLE_ID} | judge: {GEMINI_MODEL} | batch: {JUDGE_BATCH} | root: {ROOT}")

In [16]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


# A 429 is retryable when it means "too fast" and pointless when it means "out of money": backing
# off five times against a depleted balance just burns three minutes before failing anyway.
_FATAL_WORDS = ("credits are depleted", "billing", "quota exceeded for quota metric",
                "exceeded your current quota", "insufficient")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to one that repeating cannot fix."""
    msg = str(exc).lower()
    if any(w in msg for w in _FATAL_WORDS):
        return False
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in msg for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    ))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. `results=` keeps the judge and the graphs being
# judged independent — Gemini can score qwen3-14b's outputs.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout, so a parallel batch prints one block per article

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


Note: you may need to restart the kernel to use updated packages.


## 1. Load the graph and the source article

In [17]:
def load_graph(article_id):
    path = RESULTS_DIR / article_id / TRACK_DIR / GRAPH_NAME
    return json.loads(path.read_text()), path

def load_article(article_id):
    return (ARTICLE_DIR / f"{article_id}.txt").read_text()

graph, graph_path = load_graph(ARTICLE_ID)
article_text = load_article(ARTICLE_ID)

print(f"{graph_path.relative_to(ROOT)}")
print(f"  entities   {len(graph.get('entities', []))}")
print(f"  nary_nodes {len(graph.get('nary_nodes', []))}")
print(f"  relations  {len(graph.get('relations', []))}")
print(f"  article    {len(article_text):,} chars")

results/art5/e2e/nary_graph_dedup_extended_consolidated.json
  entities   72
  nary_nodes 30
  relations  163
  article    4,579 chars


## 2. Citation check — are the extractor's quotes real? (deterministic, free, **non-gating**)

Identical matching to nb9 (mojibake repair, quote/dash normalisation, elided-quote fragment walk, fuzzy
fallback), and it still resolves each quote to raw article offsets. The difference is what happens next:
in nb9 a `missing` quote **skips** the assertion, so a bad citation and an unsupported claim are the same
outcome. Here the judge sees the whole article regardless, and `evidence_status` is recorded as its own
axis — letting you separate *"the graph is wrong"* from *"the graph is right but cited the wrong line"*.

In [18]:
import re, unicodedata
from difflib import SequenceMatcher

_QUOTES = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
                         "\u2013": "-", "\u2014": "-", "\u2026": "..."})

def fix_mojibake(s: str) -> str:
    """Undo UTF-8-read-as-Latin-1 damage ("\u00c2\u00a3" -> "\u00a3") seen in some scraped article files.
    Cheap and safe: only re-decodes when the tell-tale \u00c2/\u00e2 sequences are present."""
    if not s or ("\u00c2" not in s and "\u00e3" not in s and "\u00e2" not in s):
        return s
    try:
        return s.encode("latin-1", "ignore").decode("utf-8", "ignore")
    except Exception:
        return s

def norm_text(s: str) -> str:
    """Normalise for matching: repair mojibake, unify quotes/dashes, collapse space, casefold."""
    s = fix_mojibake(s or "")
    s = unicodedata.normalize("NFKC", s).translate(_QUOTES)
    return re.sub(r"\s+", " ", s).strip().casefold()

def build_index(raw: str):
    """Normalise the article while keeping a map from every normalised char back to its RAW offset.
    That map is what lets a matched quote be resolved to real character offsets in the article."""
    raw = fix_mojibake(raw)
    chars, idx, prev_space = [], [], False
    for i, ch in enumerate(unicodedata.normalize("NFKC", raw).translate(_QUOTES)):
        if ch.isspace():
            if prev_space or not chars:
                continue
            chars.append(" "); idx.append(i); prev_space = True
        else:
            chars.append(ch.casefold()); idx.append(i); prev_space = False
    return "".join(chars), idx, raw

# The extractor sometimes elides the middle of a long quote ("Belgium recorded ... 39% more than
# usual"). Rather than failing those, split on the ellipsis and locate the fragments in order --
# the span from the first to the last fragment IS the real quote, elided middle included.
_ELLIPSIS_RE = re.compile(r"\s*(?:\.\.\.|\u2026)\s*")

def locate_quote(qn: str, art_norm: str, idx_map):
    """Find one normalised quote in the article. -> (status, ratio, (start, end) | None) in RAW offsets."""
    if not qn:
        return "missing", 0.0, None
    if qn in art_norm:
        s = art_norm.index(qn)
        return "found", 1.0, (idx_map[s], idx_map[s + len(qn) - 1] + 1)

    frags = [f for f in _ELLIPSIS_RE.split(qn) if len(f) > 3]
    if len(frags) > 1:                                   # elided quote: walk the fragments in order
        pos, first, last = 0, None, None
        for f in frags:
            i = art_norm.find(f, pos)
            if i < 0:
                first = None; break
            if first is None:
                first = i
            last = i + len(f); pos = last
        if first is not None:
            span = (idx_map[first], idx_map[last - 1] + 1)
            if span[1] - span[0] <= MAX_SPAN_CHARS:      # guard: fragments matched far apart
                return "partial", 1.0, span
            return "missing", 0.0, None

    r = best_window_ratio(qn, art_norm)
    return ("partial", r, None) if r >= FUZZY_THRESHOLD else ("missing", r, None)

def best_window_ratio(needle: str, haystack: str) -> float:
    """Best difflib ratio of `needle` against any same-length window of `haystack` (coarse stride)."""
    n = len(needle)
    if n == 0 or len(haystack) < n:
        return 0.0
    stride = max(1, n // 4)
    best = 0.0
    for i in range(0, len(haystack) - n + 1, stride):
        r = SequenceMatcher(None, needle, haystack[i:i + n]).ratio()
        if r > best:
            best = r
            if best >= 0.99:
                break
    return best

_SENT_END = re.compile(r"[.!?]")

def expand_to_sentences(raw: str, start: int, end: int):
    """Widen a span outward to sentence boundaries, so the judge sees the qualifying clauses the
    extractor may have cut ("...very high" -> "...very high compared to summer norms")."""
    left = raw.rfind(".", 0, start)
    left = 0 if left < 0 else left + 1
    m = _SENT_END.search(raw, end)
    right = len(raw) if not m else m.end()
    if right - left > MAX_SENTENCE_CHARS:                # pathological paragraph: keep the raw span
        return raw[start:end].strip()
    return raw[left:right].strip()

def flatten_evidence(ev):
    """Evidence is a str, list[str], or list[list[str]] (relations). -> flat list[str]."""
    out = []
    if ev is None:
        return out
    if isinstance(ev, str):
        return [ev]
    for item in ev:
        if isinstance(item, str):
            out.append(item)
        elif isinstance(item, list):
            out.extend(x for x in item if isinstance(x, str))
    return out

def check_evidence(evidence, art_norm, idx_map, raw):
    """-> (status, detail, resolved). `resolved` carries, per quote, its RAW offsets and the
    recovered sentence(s) -- the article's own words, not the extractor's retyped quote."""
    quotes = flatten_evidence(evidence)
    if not quotes:
        return "none", {"n_quotes": 0, "found": 0, "partial": 0, "missing": 0, "worst_ratio": None}, []

    tally = {"found": 0, "partial": 0, "missing": 0}
    worst, resolved = 1.0, []
    for q in quotes:
        st, r, span = locate_quote(norm_text(q), art_norm, idx_map)
        tally[st] += 1
        worst = min(worst, r)
        item = {"quote": q, "status": st, "ratio": round(r, 3), "start": None, "end": None,
                "matched_text": None, "sentence": None}
        if span:
            item.update({"start": span[0], "end": span[1],
                         "matched_text": raw[span[0]:span[1]].strip(),
                         "sentence": expand_to_sentences(raw, *span)})
        resolved.append(item)

    status = ("found"   if tally["missing"] == 0 and tally["partial"] == 0 else
              "partial" if tally["missing"] == 0 else "missing")
    return status, {"n_quotes": len(quotes), **tally, "worst_ratio": round(worst, 3)}, resolved

# --- the article THIS thread is working on --------------------------------------------------
# The judge prompt is built from the article text itself, so which article is "current" has to be
# per-thread: the batch cell at the end grounds several at once, and a shared global would pair one
# article's text with another's assertions. The globals are kept in step for the cells below.
import threading
_ART = threading.local()

def set_article(raw):
    """Index `raw` and make it THIS thread's current article. -> (norm, idx_map, raw)."""
    ctx = build_index(raw)
    _ART.ctx = ctx
    if threading.current_thread() is threading.main_thread():
        globals().update(ARTICLE_NORM=ctx[0], ARTICLE_IDX=ctx[1], ARTICLE_RAW=ctx[2])
    return ctx

def article_ctx():
    """(norm, idx_map, raw) for the article this thread is on; the globals when it has none."""
    return getattr(_ART, "ctx", None) or (ARTICLE_NORM, ARTICLE_IDX, ARTICLE_RAW)

ARTICLE_NORM, ARTICLE_IDX, ARTICLE_RAW = set_article(article_text)
print("article normalised:", len(ARTICLE_NORM), "chars | index map:", len(ARTICLE_IDX))

article normalised: 4578 chars | index map: 4578


In [19]:
# Run the citation check over nodes and relations (no API calls). Recorded, never gating.
def evidence_pass(graph, art_norm, idx_map, raw):
    for kind in ("nary_nodes", "relations"):
        for item in graph.get(kind, []):
            st, detail, resolved = check_evidence(item.get("evidence"), art_norm, idx_map, raw)
            item.setdefault("grounding", {}).update(
                {"evidence_status": st, "evidence_detail": detail, "evidence_resolved": resolved})
    return graph

graph = evidence_pass(graph, ARTICLE_NORM, ARTICLE_IDX, ARTICLE_RAW)

from collections import Counter
for kind in ("nary_nodes", "relations"):
    c = Counter(x["grounding"]["evidence_status"] for x in graph.get(kind, []))
    print(f"{kind:11} " + "  ".join(f"{k}={c[k]}" for k in ("found", "partial", "missing", "none") if c[k]))

bad = [(x.get("id") or f'{x.get("subject_text")} -{x.get("property")}-> {x.get("object_text")}',
        x["grounding"]["evidence_detail"])
       for kind in ("nary_nodes", "relations") for x in graph.get(kind, [])
       if x["grounding"]["evidence_status"] == "missing"]
print(f"\n{len(bad)} item(s) whose cited quote could not be found in the article "
      f"(judged anyway — the article is shown in full):")
for label, d in bad[:10]:
    print("  -", str(label)[:80], "|", d)

nary_nodes  found=30
relations   found=162  partial=1

0 item(s) whose cited quote could not be found in the article (judged anyway — the article is shown in full):


## 3. The judge — full article in, verbatim supporting quote out

One system prompt, one article, `JUDGE_BATCH` assertions per call. The judge returns, per assertion:
`supported`, `failure_type` (`intrinsic` = the article contradicts it, `extrinsic` = the article never
says it), a one-clause `note`, and — when supported — the **verbatim sentence** it relied on.

That quote is the notebook's audit trail. It is located in the article with the same matcher used on the
extractor's citations, so a judge that paraphrases or invents its supporting sentence is caught
deterministically (`quote_status`) rather than taken on trust.

In [20]:
GROUNDING_SYSTEM_PROMPT = """You are a strict grounding judge for a knowledge-graph extraction pipeline. You are given the FULL TEXT of a news article, followed by a batch of assertions extracted from it.

For each assertion decide whether THE ARTICLE SUPPORTS IT. Judge only against the article text shown. Do not use outside knowledge, and do not assume anything the article does not state, even if you believe it is true in the world.

Answer for each assertion:
- "supported": true if a careful reader of the article would agree the article asserts (or directly entails) the assertion; false otherwise.
- "failure_type": null when supported. When not supported, exactly one of:
    "intrinsic"  - the article contradicts or misstates the assertion (wrong number, wrong entity, wrong direction, overstated certainty).
    "extrinsic"  - the article simply does not say it; it may be true in the world, but this text does not state it.
- "note": one short clause naming the reason. Be specific ("article gives 2,025 not 2,000", "article never mentions the agency").
- "quote": when supported, the ONE sentence from the article that best supports the assertion, copied VERBATIM and in full from the text above - do not paraphrase it, do not shorten it, do not join two sentences. If the support genuinely needs two adjacent sentences, quote the one that carries the relation. null when not supported.

Be strict but not pedantic. Paraphrase in the ASSERTION is fine: it does not need to repeat the article's wording. Direction matters: if the article supports the reverse relation but not the stated one, that is "intrinsic". A vaguer assertion entailed by more specific text is supported; an assertion more specific than the article warrants is not.

SOME ASSERTIONS ARE ONE PART OF AN N-ARY FACT. A single real-world fact is often split across several relations sharing one hub node (e.g. the Claim "France recorded excess deaths" carries separate edges for its location, its date, and its percentage). Such an assertion is shown with a "Context - the same <hub> also asserts:" line listing the fact's SIBLING parts. Use that context to resolve what the arrow refers to - a bare "62%" or "Paris region" hanging off the France hub is that fact's Paris-region sub-figure, not a France-wide contradiction. Still rule on the single arrow shown, not on its siblings.

Return JSON only: {"results": [{"index": <int>, "supported": <bool>, "failure_type": <string|null>, "note": "<string>", "quote": "<string|null>"}]}. Return exactly one result per assertion, echoing its index."""

def verbalise_node(node):
    kind = node.get("node_kind") or "Node"
    return (f'A {kind.lower()} of type "{node.get("type")}" exists, described as '
            f'"{node.get("label")}".')

def verbalise_relation(rel):
    return (f'"{rel.get("subject_text")}" ({rel.get("subject_type")}) '
            f'--[{rel.get("property")}]--> "{rel.get("object_text")}" ({rel.get("object_type")}).')

# N-ary sibling context stays: it disambiguates what an edge MEANS, which is a different question from
# what the judge is allowed to read (here: the whole article). No evidence is pooled any more.
from collections import defaultdict

def make_relation_verbaliser(graph):
    """Verbalise a relation, appending its n-ary sibling edges as disambiguating context."""
    nodes_by_id = {n["id"]: n for n in graph.get("nary_nodes", [])}
    rels_by_subject = defaultdict(list)
    for r in graph.get("relations", []):
        rels_by_subject[r.get("subject_id")].append(r)
    def verbalise(rel):
        base = verbalise_relation(rel)
        hub = nodes_by_id.get(rel.get("subject_id"))
        if hub is None:                                  # subject is a plain entity, not a hub
            return base
        sibs = [s for s in rels_by_subject.get(rel.get("subject_id"), []) if s is not rel]
        if not sibs:
            return base
        parts = "; ".join(f'{s.get("property")} "{s.get("object_text")}"' for s in sibs)
        kind = (hub.get("node_kind") or "fact").lower()
        return (base + f'\n(Judge only the arrow above. Context - the same {kind} '
                f'"{hub.get("label")}" also asserts: {parts}.)')
    return verbalise

In [21]:
# --- The call: article + assertions -> verdicts (with the judge's own supporting quote) ----
def _verify_quote(q):
    """Locate the judge's returned sentence in the article with the SAME matcher used on the
    extractor's citations. -> (status, ratio): "found" | "partial" | "missing" | None (no quote)."""
    if not q or not str(q).strip():
        return None, None
    art_norm, art_idx, _ = article_ctx()
    st, r, _span = locate_quote(norm_text(q), art_norm, art_idx)
    return st, round(r, 3)

def _citation_match(judge_quote, cited_quotes):
    """Did the extractor cite the sentence the judge actually grounded the assertion on?
    True when some cited quote is that sentence, or a clause of it (or fuzzily equal). None when
    there is nothing to compare (unsupported, no quote, or no citation)."""
    if not judge_quote or not cited_quotes:
        return None
    jq = norm_text(judge_quote)
    if not jq:
        return None
    for c in cited_quotes:
        cn = norm_text(c)
        if not cn:
            continue
        if cn in jq or jq in cn or SequenceMatcher(None, jq, cn).ratio() >= FUZZY_THRESHOLD:
            return True
    return False

def judge_batch(items, verbalise):
    """items: list of graph dicts -> attach grounding verdicts in place. The ARTICLE is sent whole;
    nothing per-assertion is selected, so the judge — not this notebook — decides what supports what."""
    payload = [{"index": i, "assertion": verbalise(it)} for i, it in enumerate(items)]
    msg = ("ARTICLE:\n\"\"\"\n" + article_ctx()[2].strip() + "\n\"\"\"\n\n"
           "Assertions to judge:\n" + json.dumps(payload, ensure_ascii=False, indent=1))
    out = gemini_json(GROUNDING_SYSTEM_PROMPT, msg)
    by_idx = {r.get("index"): r for r in out.get("results", []) if isinstance(r, dict)}
    for i, it in enumerate(items):
        r = by_idx.get(i)
        g = it.setdefault("grounding", {})
        if r is None:                       # judge skipped it -> keep, but mark unjudged
            g.update({"supported": None, "failure_type": None, "note": "judge returned no verdict",
                      "quote": None, "quote_status": None, "citation_match": None})
            continue
        q = r.get("quote")
        qs, qr = _verify_quote(q)
        g.update({"supported": bool(r.get("supported")), "failure_type": r.get("failure_type"),
                  "note": (r.get("note") or "")[:200], "quote": q,
                  "quote_status": qs, "quote_ratio": qr,
                  "citation_match": _citation_match(q, flatten_evidence(it.get("evidence")))})
    return items

def run_judge(items, verbalise, label):
    """Every item is judged — the citation check no longer filters anything out."""
    for start in range(0, len(items), JUDGE_BATCH):
        chunk = items[start:start + JUDGE_BATCH]
        for attempt in range(1, JUDGE_RETRIES + 1):
            try:
                judge_batch(chunk, verbalise)
                break                               # success -> stop retrying
            except Exception as e:                  # transient API/JSON error -> back off and retry
                if attempt < JUDGE_RETRIES:
                    wait = JUDGE_RETRY_BACKOFF * attempt
                    print(f"  ! {label} batch {start}-{start+len(chunk)} attempt {attempt}/{JUDGE_RETRIES} "
                          f"failed: {type(e).__name__}: {e} — retrying in {wait:.0f}s")
                    time.sleep(wait)
                else:
                    print(f"  ! {label} batch {start}-{start+len(chunk)} GAVE UP: {type(e).__name__}: {e}")
                    for it in chunk:
                        it.setdefault("grounding", {}).update(
                            {"supported": None, "failure_type": None, "note": f"judge failed: {type(e).__name__}",
                             "quote": None, "quote_status": None, "citation_match": None})
        n = min(start + JUDGE_BATCH, len(items))
        print(f"  {label}: judged {n}/{len(items)}")
    return items

_n_calls = lambda n: (n + JUDGE_BATCH - 1) // JUDGE_BATCH
print(f"this article: {len(graph.get('nary_nodes', []))} nodes + {len(graph.get('relations', []))} relations "
      f"-> {_n_calls(len(graph.get('nary_nodes', [])))} + {_n_calls(len(graph.get('relations', [])))} calls "
      f"| article re-sent per call: {len(article_text):,} chars")

this article: 30 nodes + 163 relations -> 1 + 3 calls | article re-sent per call: 4,579 chars


In [22]:
# --- 3a. Ground the NODES first ------------------------------------------------------------
# A node asserts its own existence and type — nothing else in the graph tests that. And if a node
# fails, its relations must go too, so doing nodes first saves judging edges that are about to drop.
nodes = graph.get("nary_nodes", [])
print(f"grounding {len(nodes)} n-ary nodes against the full article")
run_judge(nodes, verbalise_node, "nodes")

for n in nodes:
    n["grounding"]["grounded"] = bool(n["grounding"].get("supported"))

dropped_ids = {n["id"] for n in nodes if not n["grounding"]["grounded"]}
print(f"\nnodes grounded {len(nodes) - len(dropped_ids)}/{len(nodes)}   "
      f"dropped: {sorted(dropped_ids) if dropped_ids else 'none'}")
for n in nodes:
    if not n["grounding"]["grounded"]:
        print(f'  - {n["id"]:5} {str(n.get("label"))[:52]:54} '
              f'{n["grounding"].get("failure_type")}: {n["grounding"].get("note","")[:70]}')

grounding 30 n-ary nodes against the full article
  nodes: judged 30/30

nodes grounded 30/30   dropped: none


In [23]:
# --- 3b. Ground the RELATIONS (skipping any whose endpoint node was dropped) ----------------
rels = graph.get("relations", [])
for r in rels:
    g = r.setdefault("grounding", {})
    g["orphaned"] = (r.get("subject_id") in dropped_ids) or (r.get("object_id") in dropped_ids)
    if g["orphaned"]:   # endpoint gone -> the edge cannot stand; no judge call needed
        g.update({"supported": False, "failure_type": "orphan", "note": "endpoint node failed grounding",
                  "quote": None, "quote_status": None, "citation_match": None})

live = [r for r in rels if not r["grounding"]["orphaned"]]
print(f"grounding {len(live)} relations ({len(rels)-len(live)} orphaned by dropped nodes)")
run_judge(live, make_relation_verbaliser(graph), "relations")

for r in rels:
    g = r["grounding"]
    g["grounded"] = bool(g.get("supported")) and not g["orphaned"]

print(f"\nrelations grounded {sum(r['grounding']['grounded'] for r in rels)}/{len(rels)}")
for r in rels:
    if not r["grounding"]["grounded"]:
        print(f'  - {str(r.get("subject_text"))[:26]:28} -{r.get("property"):18}-> '
              f'{str(r.get("object_text"))[:22]:24} '
              f'{r["grounding"].get("failure_type")}: {r["grounding"].get("note","")[:60]}')

grounding 163 relations (0 orphaned by dropped nodes)
  relations: judged 60/163
  relations: judged 120/163
  ! relations batch 120-163 attempt 1/3 failed: ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}} — retrying in 5s
  relations: judged 163/163

relations grounded 160/163
  - 213 people have recovered    -location          -> Democratic Republic of   extrinsic: The article does not specify the location where the 213 peop
  - Individual tested positive   -followedByEvent   -> Individual fled isolat   intrinsic: The individual fled *before* testing positive, not after.
  - Ebola treatments trial beg   -about             -> Bundibugyo virus speci   extrinsic: Article implies but does not explicitly state the trial is f


## 4. Metrics — grounding rate **and** citation precision

Three axes, deliberately kept apart:

- **grounding rate** — is the assertion supported by the article? (the headline)
- **citation precision** — of the supported assertions, how often did the extractor's own quote contain
  the sentence the judge grounded it on? A low number here with a high grounding rate means the graph is
  right but its provenance is sloppy — invisible to nb9, which picked the evidence itself.
- **judge quote integrity** — the judge's returned sentence, located in the article by the same matcher.
  Anything other than `found` means the judge paraphrased or invented its support, and its verdicts on
  those items should not be trusted.

Node rates are reported separately and should not be merged into a headline: *"a node of type X exists,
described as Y"* is close to tautological once the article mentions it, so a 100% node rate carries far
less information than the relation rate.

In [24]:
def grounding_report(graph):
    rows = {}
    for kind in ("nary_nodes", "relations"):
        items = graph.get(kind, [])
        if not items:
            continue
        g = [x.get("grounding", {}) for x in items]
        n_ok  = sum(bool(x.get("grounded")) for x in g)
        rows[kind] = {"total": len(items), "grounded": n_ok,
                      "rate": round(n_ok / len(items), 3),
                      "failures": dict(Counter(x.get("failure_type") for x in g if not x.get("grounded"))),
                      "citation_status": dict(Counter(x.get("evidence_status") for x in g))}
    # citation precision: of SUPPORTED items carrying a judge quote, how many did the extractor cite?
    for kind in ("nary_nodes", "relations"):
        items = [x.get("grounding", {}) for x in graph.get(kind, [])]
        cm = [x.get("citation_match") for x in items if x.get("grounded") and x.get("citation_match") is not None]
        if kind in rows:
            rows[kind]["citation_precision"] = (
                {"n": len(cm), "matched": sum(cm), "rate": round(sum(cm) / len(cm), 3)} if cm else None)
            qs = Counter(x.get("quote_status") for x in items if x.get("quote_status"))
            rows[kind]["judge_quote_status"] = dict(qs)
    # relation grounding split by where the PROPERTY came from — schema.org vs minted extension
    by_src = {}
    for r in graph.get("relations", []):
        src = r.get("property_source") or "unknown"
        d = by_src.setdefault(src, {"total": 0, "grounded": 0})
        d["total"] += 1; d["grounded"] += bool(r.get("grounding", {}).get("grounded"))
    for d in by_src.values():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    rows["relations_by_property_source"] = by_src
    # node grounding by type source. Claim/Observation hubs are not type-canonicalized by 6d, so they
    # carry no type_source — bucketed honestly rather than as "unknown" (which reads like missing data).
    by_t = {}
    for n in graph.get("nary_nodes", []):
        src = n.get("type_source") or "not_canonicalized"
        d = by_t.setdefault(src, {"total": 0, "grounded": 0})
        d["total"] += 1; d["grounded"] += bool(n.get("grounding", {}).get("grounded"))
    for d in by_t.values():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    rows["nodes_by_type_source"] = by_t
    return rows

report = grounding_report(graph)
print(json.dumps(report, indent=2))

{
  "nary_nodes": {
    "total": 30,
    "grounded": 30,
    "rate": 1.0,
    "failures": {},
    "citation_status": {
      "found": 30
    },
    "citation_precision": {
      "n": 30,
      "matched": 29,
      "rate": 0.967
    },
    "judge_quote_status": {
      "found": 30
    }
  },
  "relations": {
    "total": 163,
    "grounded": 160,
    "rate": 0.982,
    "failures": {
      "extrinsic": 2,
      "intrinsic": 1
    },
    "citation_status": {
      "found": 162,
      "partial": 1
    },
    "citation_precision": {
      "n": 160,
      "matched": 159,
      "rate": 0.994
    },
    "judge_quote_status": {
      "found": 159,
      "partial": 1
    }
  },
  "relations_by_property_source": {
    "schema.org": {
      "total": 103,
      "grounded": 101,
      "rate": 0.981
    },
    "vocab": {
      "total": 60,
      "grounded": 59,
      "rate": 0.983
    }
  },
  "nodes_by_type_source": {
    "vocab": {
      "total": 12,
      "grounded": 12,
      "rate": 1.0
    },
 

In [25]:
graph["grounding_report"] = report
graph["grounding_config"] = {"method": "full_article", "model": GEMINI_MODEL,
                             "judge_batch": JUDGE_BATCH, "fuzzy_threshold": FUZZY_THRESHOLD,
                             "article_chars": len(article_text)}

out_path = RESULTS_DIR / ARTICLE_ID / TRACK_DIR / OUT_NAME
out_path.write_text(json.dumps(graph, ensure_ascii=False, indent=2))
print("wrote", out_path.relative_to(ROOT))

wrote results/art5/e2e/nary_graph_grounded_fulltext.json


In [26]:
# --- Judgment log: a flat, reviewable record of EVERY grounding decision -------------------
# One row per node and per relation: the verbalised assertion, the sentence the JUDGE grounded it on
# (and whether that sentence is really in the article), what the extractor CITED, whether those two
# agree, and the verdict. JSON (full) + CSV (skim in Excel).
import csv

JUDGMENTS_JSON = "grounding_fulltext_judgments.json"
JUDGMENTS_CSV  = "grounding_fulltext_judgments.csv"

def _decision(g):
    s = g.get("supported")
    return "supported" if s else ("unjudged" if s is None else "rejected")

def _row(article_id, kind, id_, assertion, item, subject=None, prop=None, obj=None):
    g = item.get("grounding", {})
    return {"article_id": article_id, "kind": kind, "id": id_, "assertion": assertion,
            "subject": subject, "property": prop, "object": obj,
            "citation_status": g.get("evidence_status"), "orphaned": g.get("orphaned"),
            "decision": _decision(g), "supported": g.get("supported"), "grounded": g.get("grounded"),
            "failure_type": g.get("failure_type"), "justification": g.get("note"),
            "judge_quote": g.get("quote"), "judge_quote_status": g.get("quote_status"),
            "citation_match": g.get("citation_match"),
            "evidence_cited": flatten_evidence(item.get("evidence"))}

def build_judgment_log(graph, article_id):
    v_rel = make_relation_verbaliser(graph)
    rows = [_row(article_id, "node", n.get("id"), verbalise_node(n), n)
            for n in graph.get("nary_nodes", [])]
    rows += [_row(article_id, "relation",
                  f'{r.get("subject_id")}-{r.get("property")}-{r.get("object_id")}',
                  v_rel(r), r, r.get("subject_text"), r.get("property"), r.get("object_text"))
             for r in graph.get("relations", [])]
    return rows

def write_judgment_log(graph, article_id):
    rows = build_judgment_log(graph, article_id)
    d = RESULTS_DIR / article_id / TRACK_DIR
    (d / JUDGMENTS_JSON).write_text(json.dumps(rows, ensure_ascii=False, indent=2))
    cols = ["article_id","kind","id","assertion","subject","property","object","citation_status",
            "orphaned","decision","supported","grounded","failure_type","justification",
            "judge_quote","judge_quote_status","citation_match","evidence_cited"]
    with (d / JUDGMENTS_CSV).open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
        for row in rows:
            flat = dict(row)
            flat["evidence_cited"] = " || ".join(row["evidence_cited"] or [])
            w.writerow(flat)
    return rows

_jrows = write_judgment_log(graph, ARTICLE_ID)
_rej = [r for r in _jrows if r["decision"] == "rejected"]
_mis = [r for r in _jrows if r["citation_match"] is False]
_bad = [r for r in _jrows if r["judge_quote"] and r["judge_quote_status"] != "found"]
print(f"wrote {(RESULTS_DIR / ARTICLE_ID / TRACK_DIR / JUDGMENTS_JSON).relative_to(ROOT)}  (+ .csv)"
      f"  — {len(_jrows)} judgments, {len(_rej)} rejected")
print(f"\n{len(_rej)} REJECTED:")
for r in _rej[:10]:
    print(f'  [{r["kind"]:8}] {r["assertion"][:70]}')
    print(f'             -> {r["failure_type"]}: {r["justification"]}')
print(f"\n{len(_mis)} supported on a sentence the extractor did NOT cite:")
for r in _mis[:8]:
    print(f'  [{r["kind"]:8}] {r["assertion"][:66]}')
    print(f'      judge : {str(r["judge_quote"])[:110]}')
    print(f'      cited : {" || ".join(r["evidence_cited"] or [])[:110]}')
if _bad:
    print(f"\n!! {len(_bad)} judge quote(s) not found verbatim in the article — verdicts suspect:")
    for r in _bad[:5]:
        print(f'  [{r["judge_quote_status"]}] {str(r["judge_quote"])[:110]}')

wrote results/art5/e2e/grounding_fulltext_judgments.json  (+ .csv)  — 193 judgments, 3 rejected

3 REJECTED:
  [relation] "213 people have recovered" (Quantity) --[location]--> "Democratic Rep
             -> extrinsic: The article does not specify the location where the 213 people recovered.
  [relation] "Individual tested positive" (DiseaseDiagnosisEvent) --[followedByEven
             -> intrinsic: The individual fled *before* testing positive, not after.
  [relation] "Ebola treatments trial begins" (ClinicalTrialEvent) --[about]--> "Bun
             -> extrinsic: Article implies but does not explicitly state the trial is for Bundibugyo virus species.

2 supported on a sentence the extractor did NOT cite:
  [node    ] A event of type "DiseaseSpreadEvent" exists, described as "Outbrea
      judge : The current outbreak of Ebola in DR Congo began in May, though transmission had been going undetected for some
      cited : it was spreading for weeks before it was confirmed to be Ebola.

## 5. Validate the judge  ← do this before trusting any number above

A blinded 50-item sample, stratified across both verdicts so leniency and strictness are equally
testable. Fill `human` with `y`/`n` in the written file, then run the scoring cell.

The sample is drawn **across every article in `ARTICLES` that has been grounded**, and written once to
`results/<model>/_global/` — 50 items spread over N articles tests the judge far better than 50 from a
single one, and each row carries its `article_id` so you know which article to read. Before section 6
has run, only the walkthrough article can contribute; re-run the cell afterwards to pool the rest.

Judge one thing only: **does the article support this assertion?** You are shown the same assertion the
judge saw and the sentence it grounded on — but you have the article, exactly as it did, so check the
support yourself rather than trusting the quote.

Until this is filled in, the grounding rate above is an unvalidated model self-report: the extractor and
the judge are the same model.

In [ ]:
import random

SAMPLE_N = 50
random.seed(0)

def grounded_graphs():
    """The grounded graphs available right now, across the whole run: the walkthrough article from
    memory (freshest), every other one read from disk once section 6 has written it. Articles not
    yet grounded simply do not contribute — re-run this cell after the batch to pool them all."""
    out = {ARTICLE_ID: graph}
    for aid in ARTICLE_IDS:
        p = RESULTS_DIR / aid / TRACK_DIR / OUT_NAME
        if aid not in out and p.exists():
            out[aid] = json.loads(p.read_text())
    return out

def _sample_pool(article_id, g):
    v_rel = make_relation_verbaliser(g)
    return ([{"article_id": article_id, "kind": "node", "id": x.get("id"),
              "assertion": verbalise_node(x),
              "judge_quote": x["grounding"].get("quote"), "judge": x["grounding"].get("supported")}
             for x in g.get("nary_nodes", []) if "grounding" in x] +
            [{"article_id": article_id, "kind": "relation",
              "id": f'{x.get("subject_id")}-{x.get("property")}-{x.get("object_id")}',
              "assertion": v_rel(x),
              "judge_quote": x["grounding"].get("quote"), "judge": x["grounding"].get("supported")}
             for x in g.get("relations", []) if "grounding" in x])

def build_sample(graphs, n=SAMPLE_N):
    pool = [p for aid, g in graphs.items() for p in _sample_pool(aid, g)]
    yes = [p for p in pool if p["judge"] is True]
    no  = [p for p in pool if p["judge"] is False]
    random.shuffle(yes); random.shuffle(no)
    half = n // 2
    sample = yes[:half] + no[:n - half]
    if len(sample) < n:                        # top up from whichever side has more
        sample += [p for p in yes[half:] + no[n - half:]][: n - len(sample)]
    random.shuffle(sample)
    for i, s in enumerate(sample):
        s["index"] = i; s["human"] = ""        # <- fill in "y" / "n"
    return sample

# ONE sheet for the whole run, written to results/<model>/_global/ — a 50-item sample spread over
# N articles measures the judge far better than 50 items from a single one. Each row carries its
# `article_id` so you know which article to read while labelling.
_graphs = grounded_graphs()
sample = build_sample(_graphs)
(GLOBAL_DIR / "grounding_fulltext_judge_sample.json").write_text(
    json.dumps([{k: v for k, v in s.items() if k != "judge"} for s in sample], ensure_ascii=False, indent=2))
(GLOBAL_DIR / "grounding_fulltext_judge_key.json").write_text(
    json.dumps({str(s["index"]): s["judge"] for s in sample}, indent=2))
print(f"pooled {len(_graphs)}/{len(ARTICLE_IDS)} grounded article(s): {sorted(_graphs, key=_natkey)}")
print("wrote", (GLOBAL_DIR / "grounding_fulltext_judge_sample.json").relative_to(ROOT),
      f"({len(sample)} items) — fill in the `human` field, then run the next cell")

In [ ]:
# Score human labels against the judge (run after filling in `human` in the pooled sample file).
def score_agreement(sample_dir=None):
    sdir     = Path(sample_dir) if sample_dir else GLOBAL_DIR
    labelled = json.loads((sdir / "grounding_fulltext_judge_sample.json").read_text())
    key      = json.loads((sdir / "grounding_fulltext_judge_key.json").read_text())
    pairs = [(str(x["index"]), (x.get("human") or "").strip().lower()) for x in labelled]
    pairs = [(i, h) for i, h in pairs if h in ("y", "n")]
    if not pairs:
        print("no human labels yet — fill the `human` field with y/n in",
              (sdir / "grounding_fulltext_judge_sample.json").relative_to(ROOT))
        return
    agree = sum((h == "y") == bool(key[i]) for i, h in pairs)
    print(f"labelled {len(pairs)}/{len(labelled)} | agreement {agree}/{len(pairs)} = {agree/len(pairs):.1%}")
    lenient = sum(h == "n" and bool(key[i]) for i, h in pairs)     # judge said supported, human said no
    strict  = sum(h == "y" and not bool(key[i]) for i, h in pairs) # judge said no, human said supported
    print(f"  judge too lenient: {lenient}   judge too strict: {strict}")
    by_id = {str(x["index"]): x for x in labelled}
    # per-article agreement too: a pooled number can hide one article the judge reads badly
    per = {}
    for i, h in pairs:
        c = per.setdefault(by_id[i].get("article_id", "?"), [0, 0])
        c[0] += ((h == "y") == bool(key[i])); c[1] += 1
    if len(per) > 1:
        print("  by article: " + "   ".join(f"{a} {ok}/{tot}"
              for a, (ok, tot) in sorted(per.items(), key=lambda kv: _natkey(kv[0]))))
    for i, h in pairs:
        if (h == "y") != bool(key[i]):
            x = by_id[i]
            print(f"\n  [{i}] {x.get('article_id','?')}  judge={bool(key[i])} human={h}  {x['assertion'][:100]}")
            print(f"        judge quote: {str(x.get('judge_quote'))[:110]}")

score_agreement()

## 6. Batch — ground every article in `ARTICLES`

Runs the whole list resolved in the config cell (`ARTICLE_IDS`), `WORKERS` articles at a time, then
pools the per-article reports into one corpus-level rate. Sections 1–5 above are the same pipeline
walked through one article at a time for inspection; nothing here depends on having run them.

In [ ]:
def ground_article(article_id, verbose=False):
    graph, _ = load_graph(article_id)
    # bind the article to THIS thread: judge_batch builds its prompt from it, so with a parallel
    # batch a shared global would send one article's text alongside another's assertions
    art_norm, idx_map, raw = set_article(load_article(article_id))
    graph = evidence_pass(graph, art_norm, idx_map, raw)

    nodes = graph.get("nary_nodes", [])
    (run_judge(nodes, verbalise_node, "nodes") if verbose
     else _quiet(run_judge, nodes, verbalise_node, "nodes"))
    for n in nodes:
        n["grounding"]["grounded"] = bool(n["grounding"].get("supported"))
    dropped = {n["id"] for n in nodes if not n["grounding"]["grounded"]}

    rels = graph.get("relations", [])
    for r in rels:
        g = r.setdefault("grounding", {})
        g["orphaned"] = (r.get("subject_id") in dropped) or (r.get("object_id") in dropped)
        if g["orphaned"]:
            g.update({"supported": False, "failure_type": "orphan", "note": "endpoint node failed grounding",
                      "quote": None, "quote_status": None, "citation_match": None})
    live = [r for r in rels if not r["grounding"]["orphaned"]]
    v_rel = make_relation_verbaliser(graph)
    (run_judge(live, v_rel, "relations") if verbose
     else _quiet(run_judge, live, v_rel, "relations"))
    for r in rels:
        r["grounding"]["grounded"] = bool(r["grounding"].get("supported")) and not r["grounding"]["orphaned"]

    graph["grounding_report"] = grounding_report(graph)
    graph["grounding_config"] = {"method": "full_article", "model": GEMINI_MODEL,
                                 "judge_batch": JUDGE_BATCH, "fuzzy_threshold": FUZZY_THRESHOLD,
                                 "article_chars": len(raw)}
    (RESULTS_DIR / article_id / TRACK_DIR / OUT_NAME).write_text(
        json.dumps(graph, ensure_ascii=False, indent=2))
    write_judgment_log(graph, article_id)          # flat per-decision log (JSON + CSV)
    return graph["grounding_report"]

# redirect_stdout swaps sys.stdout for the WHOLE process, so under a parallel batch one article
# silencing its judge would swallow every other article's output for the duration. nb_parallel.quiet
# is the same idea, bound to the calling thread.
_quiet = nb_parallel.quiet

# The run is whatever ARTICLES resolved to in the config cell — edit it there, not here.
article_ids = ARTICLE_IDS
print(f"grounding {len(article_ids)} article(s): {article_ids}\n")

def ground_and_report(aid):
    rep = ground_article(aid)
    nd, rl = rep.get("nary_nodes", {}), rep.get("relations", {})
    cp = (rl.get("citation_precision") or {}).get("rate")
    print(f"nodes {nd.get('grounded')}/{nd.get('total')} ({nd.get('rate')})   "
          f"relations {rl.get('grounded')}/{rl.get('total')} ({rl.get('rate')})   "
          f"citation-precision {cp}   failures {rl.get('failures')}")
    return rep

# WORKERS articles at a time (WORKERS=1 in the config cell -> the plain sequential loop). Each
# article's output is buffered and printed as one block when it finishes, and a failure is reported
# and skipped rather than ending the run.
nb_parallel.reset_usage()          # the usage report at the end covers only THIS run
_res = nb_parallel.pmap(ground_and_report, article_ids, workers=WORKERS)
REPORTS = {r.item: r.value for r in _res if r.ok}

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
# wall_seconds is the batch's real clock; with WORKERS>1 the summed LLM time is larger, and the
# ratio between them is the overlap the thread pool actually bought.
nb_parallel.usage_report("nb9b · grounding (full-article judge)", wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                         write=GLOBAL_DIR / "token_usage_nb9b_grounding.json")

In [ ]:
# --- Corpus-level grounding: micro-average over every article in the run --------------------
# A single article's graph carries ~30 assertions, so its rate swings on two or three decisions.
# The number that belongs in the thesis is the pooled one: total grounded / total asserted across
# ARTICLE_IDS (micro-average — each assertion counts once, so big graphs weigh more, which is what
# "how much of what this pipeline emits is supported?" actually asks). Citation precision is pooled
# the same way, over the supported assertions that carry a judge quote.
from collections import Counter

def aggregate_reports(reports=None):
    reports = REPORTS if reports is None else reports
    agg, fails, by_prop, by_type = {}, Counter(), {}, {}
    for rep in reports.values():
        for kind in ("nary_nodes", "relations"):
            r = rep.get(kind)
            if not r:
                continue
            d = agg.setdefault(kind, {"total": 0, "grounded": 0, "cite_n": 0, "cite_matched": 0})
            d["total"] += r["total"]; d["grounded"] += r["grounded"]
            cp = r.get("citation_precision") or {}
            d["cite_n"] += cp.get("n", 0); d["cite_matched"] += cp.get("matched", 0)
            fails.update({f"{kind}:{k}": v for k, v in (r.get("failures") or {}).items() if k})
        for key, dst in (("relations_by_property_source", by_prop), ("nodes_by_type_source", by_type)):
            for src, d in (rep.get(key) or {}).items():
                t = dst.setdefault(src, {"total": 0, "grounded": 0})
                t["total"] += d["total"]; t["grounded"] += d["grounded"]
    for d in agg.values():
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
        d["citation_precision"] = round(d["cite_matched"] / d["cite_n"], 3) if d["cite_n"] else None
    for d in [*by_prop.values(), *by_type.values()]:
        d["rate"] = round(d["grounded"] / d["total"], 3) if d["total"] else None
    return {"articles": len(reports), "article_ids": sorted(reports, key=_natkey),
            **agg, "failures": dict(fails.most_common()),
            "relations_by_property_source": by_prop, "nodes_by_type_source": by_type}

AGG = aggregate_reports()
print(json.dumps({k: v for k, v in AGG.items() if k != "article_ids"}, indent=2))

# per-article table, so one bad article is visible instead of averaged away
def _cell(r):
    return f"{r.get('grounded', 0)}/{r.get('total', 0)}", f"{r.get('rate')}"

print(f"\n{'article':10} {'nodes':>10} {'rate':>7} {'relations':>12} {'rate':>7} {'cite-prec':>10}")
for aid in AGG["article_ids"]:
    rep = REPORTS[aid]
    (nc, nr), (rc, rr) = _cell(rep.get("nary_nodes", {})), _cell(rep.get("relations", {}))
    cp = (rep.get("relations", {}).get("citation_precision") or {}).get("rate")
    print(f"{aid:10} {nc:>10} {nr:>7} {rc:>12} {rr:>7} {str(cp):>10}")

summary_path = GLOBAL_DIR / "grounding_fulltext_summary.json"
summary_path.write_text(json.dumps(AGG, indent=2))
print("\nwrote", summary_path.relative_to(ROOT))

## 7. Compare with nb9 (evidence-bundle judge)

Both notebooks write their own files, so nothing is overwritten and the two runs sit side by side:

| | nb9 | nb9b (this) |
|---|---|---|
| graph | `nary_graph_grounded.json` | `nary_graph_grounded_fulltext.json` |
| log | `grounding_judgments.json` / `.csv` | `grounding_fulltext_judgments.json` / `.csv` |

The cell below diffs the two verdict sets per assertion id. Expect this notebook to be **higher** — an
assertion whose support sits a sentence away from the cited quote fails in nb9 and passes here. Read the
flips as a measure of *how much nb9's evidence window was costing*, not as a quality gain: the two
notebooks answer different questions, and this one's extra passes are exactly the population that
`citation_match=False` flags as badly cited rather than wrong.

In [ ]:
def compare_with_nb9(article_ids=None, examples=6):
    """Diff nb9's verdicts against this notebook's, over every article in the run. `examples` caps
    how many flipped assertions are printed IN TOTAL, so a 20-article corpus stays readable."""
    ids = list(article_ids or ARTICLE_IDS)
    tot = Counter()
    shown_up, shown_down = [], []
    print(f"{'article':10} {'both':>6} {'nb9':>10} {'nb9b':>10} {'+':>5} {'-':>5}")
    for aid in ids:
        d = RESULTS_DIR / aid / TRACK_DIR
        p9, p9b = d / "grounding_judgments.json", d / JUDGMENTS_JSON
        if not p9.exists() or not p9b.exists():
            print(f"{aid:10} skipped — {'nb9' if not p9.exists() else 'nb9b'} log missing")
            continue
        a = {r["id"]: r for r in json.loads(p9.read_text())}
        b = {r["id"]: r for r in json.loads(p9b.read_text())}
        both = [i for i in b if i in a]
        n_a  = sum(a[i]["grounded"] for i in both)
        n_b  = sum(b[i]["grounded"] for i in both)
        up   = [i for i in both if not a[i]["grounded"] and b[i]["grounded"]]
        down = [i for i in both if a[i]["grounded"] and not b[i]["grounded"]]
        tot.update({"both": len(both), "nb9": n_a, "nb9b": n_b, "up": len(up), "down": len(down),
                    "articles": 1})
        print(f"{aid:10} {len(both):>6} {n_a:>10} {n_b:>10} {len(up):>5} {len(down):>5}")
        shown_up   += [(aid, a[i], b[i]) for i in up]
        shown_down += [(aid, b[i]) for i in down]

    if not tot["articles"]:
        print("nothing to compare — run 9_grounding.ipynb over the same ARTICLES first")
        return
    n = tot["both"]
    print(f"\npooled over {tot['articles']} article(s): {n} assertions in both logs")
    print(f"  nb9  grounded {tot['nb9']}/{n} ({tot['nb9']/n:.1%})")
    print(f"  nb9b grounded {tot['nb9b']}/{n} ({tot['nb9b']/n:.1%})")
    print(f"  rejected by nb9, supported here: {tot['up']}   |   opposite: {tot['down']}")

    for aid, ra, rb in shown_up[:examples]:
        print(f"\n  + [{aid}] {rb['assertion'][:84]}")
        print(f"      nb9  : {ra['failure_type']}: {str(ra['justification'])[:90]}")
        print(f"      nb9b : grounded on {str(rb['judge_quote'])[:90]}")
        print(f"      cited by extractor? {rb['citation_match']}")
    for aid, rb in shown_down[:max(1, examples // 2)]:
        print(f"\n  - [{aid}] {rb['assertion'][:84]}")
        print(f"      nb9b : {rb['failure_type']}: {str(rb['justification'])[:90]}")

compare_with_nb9()

## 8. Judge benchmark — 250 real assertions + 250 corrupted ones

Section 5 samples the judge's own output, so it can only ask *"when the judge says supported, is
it?"*. It cannot answer the question a grounding claim actually rests on: **would this judge catch a
fact the extractor got wrong?** The judge grounds ~98% of what it sees, so a sample of its verdicts
contains almost no true negatives, and any recall number read off it rests on a handful of items.

This section builds the missing half. `judge_benchmark.py` takes the graphs of the whole run and
produces 500 items:

| half | how it is made | gold label | what it measures |
|---|---|---|---|
| 250 **clean** | drawn as-is, proportional per article | **you label it** — most are supported; the rest are the extractor's real errors | false-positive rate: good facts the judge rejects |
| 250 **corrupted** | one deterministic meaning-changing edit each, *verified absent from the article* | not supported, by construction | recall on bad facts, broken down by the kind of error |

Seven corruption operators, each a different way a pipeline actually goes wrong: `object_swap` /
`subject_swap` (right structure, wrong entity — what a canonicalisation step produces), `number_edit`
(a quantity or date moved off what the article says), `hedge_flip` ("more than" → "fewer than",
"suspected" → "confirmed"), `property_swap` (a property the corpus never uses with this object
type), `direction_flip` (the arrow reversed), `type_change` (an unrelated type). The two halves are
drawn from **disjoint** base assertions, so no item in the sheet is the recognisable twin of another.

**The corrupted half is a hypothesis, not a fact.** An edit can land on something the article
supports anyway, so those items carry a `human` column too — `score()` separates *"the judge missed
it"* from *"the corruption did not corrupt"* and reports the second count rather than charging it to
the judge.

Everything here reads the judge defined above — same prompt, same verbalisers, same quote check — so
the benchmark always measures the judge this notebook would actually run.

In [ ]:
# --- 8a. Build the benchmark (deterministic, no API calls) ---------------------------------
# Written to results/<RESULTS_MODEL>/_global/. build() REFUSES to overwrite an existing sheet:
# rebuilding renumbers every item, which would orphan your labels and every verdict file scored
# against them. Pass overwrite=True only to start over.
import importlib, judge_benchmark as jb
importlib.reload(jb)

BENCH_ARTICLES = ARTICLE_IDS                    # the whole run; the sheet is grouped by article

bench_graphs   = {aid: load_graph(aid)[0] for aid in BENCH_ARTICLES}
bench_articles = {aid: load_article(aid)  for aid in BENCH_ARTICLES}

bench_items = jb.build(bench_graphs, bench_articles, GLOBAL_DIR,
                       verbalise_node, make_relation_verbaliser, norm_text,
                       n_clean=250, n_corrupt=250, source=GRAPH_NAME)

In [ ]:
# --- 8a′. Read a few corruptions before trusting them --------------------------------------
# Every edit is verified absent from its article, which makes the gold label defensible — it does
# not make the item readable. Skim one of each operator: an edit that produced word salad is a
# negative any judge catches for the wrong reason, and inflates recall.
_key = jb.load_json(GLOBAL_DIR / jb.KEY_FILE)
_by_i = {x["index"]: x for x in bench_items}
_seen = set()
for i, k in _key.items():
    if k["origin"] != "corrupt" or k["op"] in _seen:
        continue
    _seen.add(k["op"])
    print(f'\n[{i}] {k["article_id"]}  {k["op"]}   ({k["edit"]})')
    print("   before:", k["original_assertion"].splitlines()[0][:120])
    print("   after :", _by_i[int(i)]["assertion"].splitlines()[0][:120])

In [ ]:
# --- 8b. Run the judge over the 500 items ---------------------------------------------------
# One verdict file per judge run, named by TAG. A tag is the judge AND its sampling settings: two
# seeds of the same model are two runs, and the gap between them is this judge's reproducibility
# (compare() prints it). The prompt is hashed into the file too — a benchmark result belongs to a
# prompt, and silently editing section 3's prompt would otherwise invalidate it invisibly.
import hashlib

# The registry, not llm_backend's module globals: a Gemini backend never calls configure(), so
# those still hold their import-time defaults (THINKING=True) and would mislabel the run.
_SPEC = llm_backend.BACKENDS[BACKEND]

JUDGE_BATCH = 20 if _SPEC.get("thinking") else JUDGE_BATCH
BENCH_TAG   = BACKEND + (f"_seed{llm_backend.SEED}" if llm_backend.SEED is not None else "")
# BENCH_TAG = BACKEND + "_seed4321_run2"        # <- edit for a repeat run; never overwrite a tag

bench_items    = jb.load_items(GLOBAL_DIR)      # canonical items, with any labels merged in
bench_articles = {aid: load_article(aid) for aid, _ in jb.by_article(bench_items)}
bench_groups   = dict(jb.by_article(bench_items))

def judge_bench_article(aid):
    set_article(bench_articles[aid])             # thread-local: the judge prompt is built from it
    nb_parallel.quiet(run_judge, bench_groups[aid], jb.assertion_of, aid)
    return sum(x.get("grounding", {}).get("supported") is not None for x in bench_groups[aid])

nb_parallel.reset_usage()
_res = nb_parallel.pmap(judge_bench_article, list(bench_groups), workers=WORKERS)

jb.save_verdicts(bench_items, GLOBAL_DIR, BENCH_TAG, config={
    "backend": BACKEND, "model": GEMINI_MODEL, **{k: v for k, v in _SPEC.items() if k not in ("kind", "model")},
    "temperature": llm_backend.TEMPERATURE, "top_p": llm_backend.TOP_P, "seed": llm_backend.SEED,
    "judge_batch": JUDGE_BATCH, "graph": GRAPH_NAME, "results_model": RESULTS_MODEL,
    "prompt_sha1": hashlib.sha1(GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]})

nb_parallel.usage_report(f"nb9b · judge benchmark ({BENCH_TAG})",
                         wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                         write=GLOBAL_DIR / f"token_usage_judge_benchmark_{BENCH_TAG}.json")

In [ ]:
# --- 8b-repeat. The same 500 items, judged N times ------------------------------------------
# Reproducibility is a measurement here, not an assumption. vLLM batches requests continuously, so
# batch shape moves logits in the low bits and can flip argmax wherever the top two tokens are
# near-tied — a fixed seed makes a run RE-RUNNABLE, not determined. These runs hold everything
# fixed (same seed, same WORKERS, same server) so the spread between them IS the noise floor: no
# difference between two judges that is smaller than it means anything.
#
# Keep WORKERS identical across the runs. Changing it changes the server's batching, which is one
# of the things being measured.
import hashlib

_SPEC = llm_backend.BACKENDS[BACKEND]
JUDGE_BATCH = 20 if _SPEC.get("thinking") else JUDGE_BATCH
BENCH_SEED  = 1234          # same seed in every run -> this tests re-runnability
BENCH_RUNS  = 3
llm_backend.configure(seed=BENCH_SEED)

for _run in range(1, BENCH_RUNS + 1):
    _tag = f"{BACKEND}_seed{BENCH_SEED}_run{_run}"
    if (GLOBAL_DIR / jb.VERDICTS_FILE.format(tag=_tag)).exists():
        print(f"{_tag}: already done, skipping")      # safe to re-run the cell after a crash
        continue
    _items  = jb.load_items(GLOBAL_DIR)               # fresh dicts: no verdicts carried between runs
    _groups = dict(jb.by_article(_items))
    _arts   = {a: load_article(a) for a in _groups}

    def _judge_one(aid, _g=_groups, _a=_arts):
        set_article(_a[aid])                          # thread-local: the prompt is built from it
        nb_parallel.quiet(run_judge, _g[aid], jb.assertion_of, aid)
        return sum(x.get("grounding", {}).get("supported") is not None for x in _g[aid])

    print(f"\n=== run {_run}/{BENCH_RUNS}  ({_tag}) ===")
    nb_parallel.reset_usage()
    nb_parallel.pmap(_judge_one, list(_groups), workers=WORKERS)
    jb.save_verdicts(_items, GLOBAL_DIR, _tag, config={
        "backend": BACKEND, "model": GEMINI_MODEL,
        **{k: v for k, v in _SPEC.items() if k not in ("kind", "model")},
        "temperature": llm_backend.TEMPERATURE, "top_p": llm_backend.TOP_P,
        "seed": llm_backend.SEED, "judge_batch": JUDGE_BATCH, "workers": WORKERS,
        "graph": GRAPH_NAME, "results_model": RESULTS_MODEL, "repeat": _run,
        "prompt_sha1": hashlib.sha1(GROUNDING_SYSTEM_PROMPT.encode()).hexdigest()[:12]})
    nb_parallel.usage_report(f"nb9b · benchmark {_tag}",
                             wall_seconds=nb_parallel.LAST_WALL_SECONDS,
                             write=GLOBAL_DIR / f"token_usage_judge_benchmark_{_tag}.json")

### 8c. Label, then score

Open `judge_benchmark_sheet.csv` (grouped by article, so you read each article once) and fill the
`human` column with `y` / `n` — *does this article support this assertion?* — using `human_note` for
anything you want to justify later. Do **not** open `judge_benchmark_key.json` while labelling; it
holds the gold labels and the provenance of every edit.

An efficient protocol, if labelling all 500 is more than you want to do:

1. Label **all 250 clean items**. This half is what the false-positive rate is computed from, and
   nothing else can supply it.
2. Leave the corrupted half blank — they are gold-negative by construction. Then run `score()` and
   **label only the corrupted items it lists as MISSED**. Each one is either a judge failure or a
   corruption that failed to corrupt, and only reading the article tells you which. Marking a failed
   corruption `y` excludes it from the metrics instead of charging it to the judge.

`score()` reports recall on the corrupted half with a Wilson interval, the false-positive rate on
the clean half, and — the number worth putting in the thesis — **recall per corruption type**. A
judge can look strong overall and be blind to one class; number edits are usually the one.

In [ ]:
jb.score(GLOBAL_DIR, BENCH_TAG)

In [ ]:
# --- 8d. Every judge run side by side -------------------------------------------------------
# Two seeds of one model bound its reproducibility; two models are the actual choice of judge. Both
# read off the same frozen 500 items, which is the whole point of freezing them.
jb.compare(GLOBAL_DIR)